# Router fine-tune: Qwen2.5-1.5B-Instruct + QLoRA

Fine-tunes a 1.5B open model to classify customer messages into `support | orders | escalation`, then compares **base vs fine-tuned vs a large API model** on a held-out test set (accuracy, latency, cost).

Runtime: **GPU (T4)** - Runtime > Change runtime type > T4 GPU. Takes about 10 minutes end to end.

In [ ]:
# 1. Get the code
REPO = 'https://github.com/khizerabbas01/cadral-ai-ops-agent.git'
!git clone -q $REPO repo
%cd repo
!pip -q install -r finetune/requirements.txt

In [ ]:
# 2. Build the dataset (train/val/test with no template overlap between train and test)
!python finetune/build_dataset.py

In [ ]:
# 3. Fine-tune (QLoRA, rank 16, 3 epochs)
!python finetune/train.py --out finetune/out/router-lora

In [ ]:
# 4. Evaluate base vs fine-tuned. Add a free Groq key to include the large API model in the comparison.
import os
from google.colab import userdata
try:
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
    extra = '--api-model openai/gpt-oss-120b'
except Exception:
    extra = ''
!python finetune/evaluate.py --adapter finetune/out/router-lora $extra

In [ ]:
# 5. Results table (copy finetune/results/ back into the repo)
print(open('finetune/results/results.md').read())

In [ ]:
# 6. Optional: download the adapter to serve it (finetune/serve.py) behind CLASSIFIER_URL
!zip -qr router-lora.zip finetune/out/router-lora finetune/results
from google.colab import files
files.download('router-lora.zip')